# LangChain Agents, Reasoning Loops & Model Context Protocol (MCP)

In this notebook, we transition from **deterministic chains** to **autonomous AI Agents**. An agent uses a language model as a reasoning engine to dynamically determine which actions to take, execute those actions via external tools, observe the results, and repeat until the objective is accomplished.

### Notebook Roadmap (Progression by Complexity):
1. **Agent 1: Simple ReAct Agent** — **Customer Support & Order Resolution** — Building a baseline Reasoning + Acting agent with Google Gemini, custom `@tool` functions (`lookup_customer_order`, `calculate_refund_and_restocking`, `format_support_ticket_receipt`), compiled directly via modern `create_agent`.
2. **Agent 2: Intermediate Resilient Agent** — **Real Estate & Apartment Rental Finder** — Combining custom enterprise database tools (`query_rental_listings_database`) with native web search tools (`search_online_listings`) and rent affordability calculations (`calculate_rent_affordability`), featuring self-correcting error trapping with `handle_tool_error=True`.
3. **Agent 3: Advanced Structured Conversational Agent** — **Healthcare & Pharmacy Prescription Assistant** — Utilizing multi-argument Pydantic input schemas (`args_schema`) for pediatric dosage calculations and drug allergy interaction checking, paired with conversational message state management.
4. **Special Feature: Model Context Protocol (MCP) Integration** — Translating standardized MCP tool definitions into native LangChain tools using a production-grade `MCPToolAdapter` for clinic appointment scheduling.

### 1. Environment Configuration & Google Gemini LLM Initialization

Before assembling agents, we load environment variables from `.env` and initialize the Google Gemini Chat Model (`ChatGoogleGenerativeAI`). We set `temperature=0` to ensure deterministic, precise reasoning during tool invocation and schema compliance.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables
load_dotenv()

# Initialize Google Gemini 3.6 Flash as our reasoning LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=0.0,
    max_retries=3
)

print(f"Initialized LLM successfully: {llm.model}")

---
## Agent 1: Simple ReAct Agent (Customer Support & Order Resolution)

The **ReAct** (Reasoning + Acting) framework allows the agent to iteratively call tools and incorporate observations into its message state:
```
User Query -> Thought -> Tool Call -> Observation -> Thought -> Final Answer
```

In this section, we create three specialized `@tool` functions for customer service:
- `lookup_customer_order`: Queries order database for status and items.
- `calculate_refund_and_restocking`: Calculates refund amount after deducting restocking fee.
- `format_support_ticket_receipt`: Formats a formal customer resolution ticket.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_customer_order(order_id: str) -> str:
    """
    Looks up customer order details from the e-commerce database.
    Args:
        order_id: The unique order identifier (e.g., 'ORD-8821', 'ORD-4402').
    Returns:
        Order details including customer name, purchased items, status, and total amount.
    """
    orders_db = {
        "ORD-8821": "Customer: Sarah Connor | Items: [Wireless Noise-Canceling Headphones] | Status: Delivered (3 days ago) | Total: $180.00 | Return Eligible: Yes",
        "ORD-4402": "Customer: Alex Murphy | Items: [Mechanical Gaming Keyboard, USB-C Hub] | Status: Delivered (12 days ago) | Total: $145.00 | Return Eligible: Yes",
        "ORD-9910": "Customer: Bruce Wayne | Items: [4K Ultra-Wide Monitor] | Status: In Transit | Total: $650.00 | Return Eligible: No"
    }
    order_key = order_id.strip().upper()
    return orders_db.get(order_key, f"Order '{order_key}' not found in order database.")


@tool
def calculate_refund_and_restocking(order_total: float, return_reason: str = "defective") -> str:
    """
    Calculates the applicable restocking fee and final refund amount for a returned item.
    Args:
        order_total: The total dollar amount of the purchase (e.g. 180.00).
        return_reason: Reason for return ('defective', 'wrong_item', 'change_of_mind').
    Returns:
        A breakdown of restocking fee percentage, fee amount, and net refund.
    """
    reason = return_reason.lower().strip()
    if reason in ["defective", "wrong_item"]:
        restock_fee_pct = 0.0
    elif reason == "change_of_mind":
        restock_fee_pct = 10.0
    else:
        restock_fee_pct = 5.0
        
    fee_amount = order_total * (restock_fee_pct / 100.0)
    net_refund = order_total - fee_amount
    return (
        f"Order Total: ${order_total:.2f} | Reason: {reason} | "
        f"Restocking Fee ({restock_fee_pct}%): ${fee_amount:.2f} | Net Refund: ${net_refund:.2f}"
    )


@tool
def format_support_ticket_receipt(customer_name: str, resolution_summary: str, priority_level: str = "Standard") -> str:
    """
    Formats an official Customer Support Resolution Ticket.
    Args:
        customer_name: Full name of the customer.
        resolution_summary: Summary of the order lookup and refund calculation.
        priority_level: Ticket priority ('Standard', 'Urgent', 'VIP').
    Returns:
        Standardized support ticket string.
    """
    return (
        f"=== HONRA CUSTOMER SUPPORT RESOLUTION TICKET ===\n"
        f"CUSTOMER: {customer_name}\n"
        f"PRIORITY: {priority_level}\n"
        f"RESOLUTION: {resolution_summary}\n"
        f"STATUS: Verified & Processed by Autonomous ReAct Agent\n"
        f"================================================"
    )

# Configure tool resilience
lookup_customer_order.handle_tool_error = True
calculate_refund_and_restocking.handle_tool_error = True
format_support_ticket_receipt.handle_tool_error = True

agent1_tools = [lookup_customer_order, calculate_refund_and_restocking, format_support_ticket_receipt]
print("Agent 1 Tools Registered:", [t.name for t in agent1_tools])

### 1.2 Defining and Executing Agent 1 with `create_agent`

In LangChain 1.0, agents are assembled using **`create_agent`** from `langchain.agents`. It automatically compiles the model and tool bindings into an efficient executable state graph. We invoke the agent with a `messages` dictionary.

In [ ]:
from langchain.agents import create_agent

# Build and compile Agent 1
agent1 = create_agent(
    model=llm,
    tools=agent1_tools,
    system_prompt=(
        "You are a helpful customer support agent for an e-commerce platform. "
        "Answer user inquiries by looking up orders, calculating refund/restocking fees, "
        "and formatting official resolution receipts."
    )
)

# Execute query requiring multi-step tool chaining
query1 = (
    "Look up order 'ORD-8821'. If the customer wants to return the item due to 'change_of_mind', "
    "calculate the net refund after the restocking fee, and generate a formal support ticket receipt for Sarah Connor."
)

print("--- Executing Agent 1 (Customer Support ReAct) ---")
result1 = agent1.invoke({"messages": [("user", query1)]})

# Extract final assistant response from message state
final_message1 = result1["messages"][-1]
print("\n--- Agent 1 Output ---\n", final_message1.content)

---
## Agent 2: Intermediate Resilient Agent (Real Estate & Apartment Rental Finder)

In real-world applications, external tools can fail due to database lookup misses or missing records. Without resilience, an unhandled exception crashes the entire agent.

With **`handle_tool_error=True`**, LangChain catches the `ToolException` and passes the error text back to the model as a `ToolMessage` observation. This enables the agent to inspect the failure, self-correct, and choose an alternative path (e.g. falling back to live web search or querying alternative neighborhoods).

### Tools Registered:
1. `query_rental_listings_database`: Internal rental listing database (simulates `ToolException` for unlisted areas).
2. `search_online_listings`: Native `@tool` web search powered by DuckDuckGo (`duckduckgo_search`).
3. `calculate_rent_affordability`: Computes rent-to-income ratio against the standard 30% rule.

In [ ]:
from langchain_core.tools import tool, ToolException
from duckduckgo_search import DDGS

# 1. Resilient Web Search Tool using native @tool and duckduckgo_search
@tool
def search_online_listings(query: str) -> str:
    """
    Searches the live web for real estate rental apartment listings and market pricing trends.
    Args:
        query: Search keywords (e.g., 'Sunnyvale 2 bedroom apartment rent prices').
    Returns:
        Summarized search result snippets with listing details.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return f"No online rental listings found for '{query}'."
            snippets = [f"- [{r.get('title', 'Listing')}] {r.get('body', '')}" for r in results]
            return "\n".join(snippets)
    except Exception as e:
        raise ToolException(f"Web Search Engine Error: {str(e)}")

search_online_listings.handle_tool_error = True

# 2. Internal Rental Listings Database with deliberate error simulation
@tool
def query_rental_listings_database(neighborhood: str) -> str:
    """
    Queries the internal property database for available apartment listings in a neighborhood.
    Throws a ToolException if the neighborhood is not indexed in the internal database.
    """
    neighborhood_clean = neighborhood.strip().lower()
    listings_db = {
        "downtown": "Downtown Loft: 2 Bed / 2 Bath | $2,800/mo | Amenities: Gym, Concierge, Metro access",
        "sunnyvale": "Sunnyvale Garden Apartment: 2 Bed / 1 Bath | $2,100/mo | Amenities: Pool, Parking, Near Parks",
        "greenwich": "Greenwich Townhouse: 3 Bed / 2.5 Bath | $3,500/mo | Amenities: Private Garden, In-unit Laundry"
    }
    if neighborhood_clean in listings_db:
        return listings_db[neighborhood_clean]
    else:
        raise ToolException(f"Database Error: No rental listings indexed for '{neighborhood}'. Fallback: Search online or check 'Downtown' / 'Sunnyvale'.")

query_rental_listings_database.handle_tool_error = True

# 3. Rent Affordability Calculator Tool
@tool
def calculate_rent_affordability(monthly_rent: float, monthly_income: float) -> str:
    """
    Calculates the rent-to-income percentage ratio and evaluates whether it meets the standard 30% affordability rule.
    """
    if monthly_income <= 0:
        raise ToolException("Income Error: Monthly income must be greater than zero.")
    ratio_pct = (monthly_rent / monthly_income) * 100.0
    recommendation = "AFFORDABLE (Within 30% rule)" if ratio_pct <= 30.0 else "STRETCH / OVER-BUDGET (Exceeds 30% rule)"
    return f"Rent: ${monthly_rent:,.2f}/mo | Income: ${monthly_income:,.2f}/mo | Rent-to-Income Ratio: {ratio_pct:.1f}% | Evaluation: {recommendation}"

calculate_rent_affordability.handle_tool_error = True

agent2_tools = [query_rental_listings_database, search_online_listings, calculate_rent_affordability]
print("Agent 2 Tools Registered:", [t.name for t in agent2_tools])

### 2.2 Assembling and Testing Agent 2 Resilience

We construct Agent 2 and execute a prompt that deliberately requests listings in an unindexed area (`Uptown`), followed by known neighborhoods. We observe how the agent traps the `ToolException`, inspects the error observation, and smoothly continues to solve the query.

In [ ]:
from langchain.agents import create_agent

agent2 = create_agent(
    model=llm,
    tools=agent2_tools,
    system_prompt=(
        "You are a resilient real estate assistant. Help users search property listings and evaluate "
        "rent affordability. If a database query fails for a neighborhood, inspect the error observation, "
        "and query available alternatives or use web search to fulfill the user request."
    )
)

# Query designed to trigger database miss then recover and calculate affordability
query_resilience = (
    "A client earns $8,000 monthly income and wants an apartment. First check the internal listings for 'Uptown'. "
    "If that fails, check 'Sunnyvale' and 'Downtown', and calculate whether the Sunnyvale rent is affordable under their income."
)

print("--- Executing Agent 2 (Resilient Real Estate Agent) ---")
result2 = agent2.invoke({"messages": [("user", query_resilience)]})
final_message2 = result2["messages"][-1]
print("\n--- Agent 2 Output ---\n", final_message2.content)

---
## Agent 3: Advanced Structured Conversational Agent (Healthcare & Pharmacy Assistant)

In enterprise applications, tools frequently accept multiple structured parameters with strict constraints (e.g. patient weight, age, dosage frequencies, known allergies).

### Key Capabilities:
1. **Pydantic `args_schema`**: Enforces type safety, minimum/maximum ranges, and field validation on multi-argument tool calls before the function executes.
2. **Conversational Message State**: Preserves context across multi-turn interactions by passing previous message turns directly in the `messages` list.

In [ ]:
from typing import Literal, Optional, List
from pydantic import BaseModel, Field
from langchain_core.tools import tool, ToolException

# 1. Define Strict Pydantic Input Schema for Medication Dosage Tool
class PediatricDosageInput(BaseModel):
    patient_weight_kg: float = Field(
        ..., 
        gt=0,
        description="Weight of the patient in kilograms (must be strictly > 0)."
    )
    medication_name: str = Field(
        ...,
        description="Name of the medication (e.g., 'Amoxicillin', 'Ibuprofen', 'Paracetamol')."
    )
    dosage_mg_per_kg: float = Field(
        default=15.0,
        gt=0,
        description="Recommended single dose in mg per kg body weight."
    )
    frequency_hours: int = Field(
        default=8,
        ge=4, le=24,
        description="Interval between doses in hours (between 4 and 24 hours)."
    )

@tool(args_schema=PediatricDosageInput)
def calculate_pediatric_dosage(patient_weight_kg: float, medication_name: str, 
                              dosage_mg_per_kg: float = 15.0, frequency_hours: int = 8) -> str:
    """
    Calculates the exact single dose (mg) and total daily maximum dose for pediatric prescriptions.
    """
    single_dose_mg = patient_weight_kg * dosage_mg_per_kg
    doses_per_day = 24 // frequency_hours
    daily_total_mg = single_dose_mg * doses_per_day
    
    return (
        f"Prescription Calculation for {medication_name}:\n"
        f"- Patient Weight: {patient_weight_kg:.1f} kg\n"
        f"- Single Dose: {single_dose_mg:.1f} mg (administered every {frequency_hours} hours)\n"
        f"- Daily Total ({doses_per_day} administrations): {daily_total_mg:.1f} mg/day"
    )

calculate_pediatric_dosage.handle_tool_error = True

# 2. Define Schema for Drug Interaction & Allergy Checker
class DrugAllergyCheckInput(BaseModel):
    medication_name: str = Field(..., description="Medication to check.")
    known_allergies: List[str] = Field(default_factory=list, description="List of patient known drug allergies.")

@tool(args_schema=DrugAllergyCheckInput)
def check_drug_allergy_conflict(medication_name: str, known_allergies: List[str]) -> str:
    """
    Checks if a target medication conflicts with any documented patient allergies.
    """
    med_clean = medication_name.strip().lower()
    allergies_clean = [a.strip().lower() for a in known_allergies]
    
    # Allergy conflict database
    penicillin_family = ["penicillin", "amoxicillin", "ampicillin", "augmentin"]
    nsaid_family = ["ibuprofen", "aspirin", "naproxen"]
    
    conflicts = []
    for allergy in allergies_clean:
        if allergy in penicillin_family and med_clean in penicillin_family:
            conflicts.append(f"CRITICAL: {medication_name} is in the Penicillin class and conflicts with allergy '{allergy}'.")
        elif allergy in nsaid_family and med_clean in nsaid_family:
            conflicts.append(f"WARNING: {medication_name} is an NSAID and conflicts with allergy '{allergy}'.")
            
    if conflicts:
        return " | ".join(conflicts) + " Recommendation: DO NOT DISPENSE. Substitute with alternative class."
    return f"Safety Check PASSED: {medication_name} has no detected conflicts with listed allergies {known_allergies}."

check_drug_allergy_conflict.handle_tool_error = True

agent3_tools = [calculate_pediatric_dosage, check_drug_allergy_conflict]
print("Agent 3 Tools Registered with Pydantic Schemas:", [t.name for t in agent3_tools])

### 3.2 Conversational Reasoning Across Multi-Turn Healthcare Interactions

In modern LangChain, multi-turn state is preserved by appending prior message turns into the `messages` list for subsequent invocations.

In [ ]:
from langchain.agents import create_agent

agent3 = create_agent(
    model=llm,
    tools=agent3_tools,
    system_prompt=(
        "You are a certified Clinical Pharmacy Assistant. Calculate accurate pediatric medication dosages "
        "and verify drug allergy conflicts across conversational turns using your available tools."
    )
)

# Multi-turn conversation simulation
# Turn 1: Initial prescription dosage query
turn1_input = "A 5-year-old child weighing 20.0 kg requires Amoxicillin at 20 mg/kg every 8 hours. Calculate the dosage per administration and daily total."
print("=== Turn 1 Query ===")
res_turn1 = agent3.invoke({"messages": [("user", turn1_input)]})
print("\n--- Turn 1 Result ---\n", res_turn1["messages"][-1].content)

# Turn 2: Follow-up query requiring context from Turn 1
turn2_input = "The patient's chart states a documented allergy to 'Penicillin'. Based on the medication calculated in the previous step, check if there is an allergy conflict."
print("\n=== Turn 2 Query (Contextual Follow-up) ===")
res_turn2 = agent3.invoke({"messages": res_turn1["messages"] + [("user", turn2_input)]})
print("\n--- Turn 2 Result ---\n", res_turn2["messages"][-1].content)

---
## Extra Special Feature: Model Context Protocol (MCP) Tool Integration

### What is the Model Context Protocol (MCP)?
The **Model Context Protocol (MCP)** is an open standard designed to solve tool fragmentation. Instead of writing custom point-to-point wrappers for every API, an MCP Server exposes a standard JSON-RPC interface with three primitives:
1. **Tools**: Callable functions with standardized JSON schema parameter contracts.
2. **Resources**: Read-only contextual data (files, database snapshots, logs).
3. **Prompts**: Parameterized reusable prompts.

Below, we implement an **`MCPToolAdapter`** that dynamically converts an MCP clinic appointment tool specification into a native LangChain `StructuredTool` and binds it into an active LangChain agent.

In [ ]:
from typing import Any, Callable, Dict, Optional
from langchain_core.tools import StructuredTool
from pydantic import create_model, Field
from langchain.agents import create_agent

class MCPToolAdapter:
    """
    Production adapter that bridges Model Context Protocol (MCP) tool schemas into LangChain StructuredTools.
    """
    @staticmethod
    def convert_mcp_to_langchain(
        mcp_schema: Dict[str, Any], 
        execute_fn: Callable[..., Any]
    ) -> StructuredTool:
        """
        Converts an MCP tool definition (name, description, inputSchema) into a LangChain StructuredTool.
        """
        tool_name = mcp_schema["name"]
        tool_desc = mcp_schema.get("description", "")
        json_schema = mcp_schema.get("inputSchema", {})
        properties = json_schema.get("properties", {})
        required = json_schema.get("required", [])

        type_mapping = {
            "string": str,
            "integer": int,
            "number": float,
            "boolean": bool,
            "array": list,
            "object": dict
        }

        field_definitions = {}
        for prop_name, prop_meta in properties.items():
            py_type = type_mapping.get(prop_meta.get("type", "string"), str)
            desc = prop_meta.get("description", "")
            if prop_name in required:
                field_definitions[prop_name] = (py_type, Field(..., description=desc))
            else:
                default_val = prop_meta.get("default", None)
                field_definitions[prop_name] = (Optional[py_type], Field(default=default_val, description=desc))

        pydantic_model = create_model(f"{tool_name.capitalize()}Input", **field_definitions)

        return StructuredTool.from_function(
            func=execute_fn,
            name=tool_name,
            description=tool_desc,
            args_schema=pydantic_model,
            handle_tool_error=True
        )


# 1. Define standard MCP Tool Contract for Clinic Appointments
mcp_clinic_appointment_spec = {
    "name": "mcp_schedule_clinic_appointment",
    "description": "MCP Server Tool: Books urgent care or specialist appointments in the hospital clinic system.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "patient_name": {"type": "string", "description": "Full name of the patient"},
            "department": {"type": "string", "description": "Clinic department (e.g., 'pediatrics', 'general_care', 'cardiology')"},
            "preferred_time": {"type": "string", "description": "Preferred time window (e.g., 'morning', 'afternoon')", "default": "morning"}
        },
        "required": ["patient_name", "department"]
    }
}

# 2. Simulated MCP Server RPC Handler
def handle_mcp_clinic_rpc(patient_name: str, department: str, preferred_time: str = "morning") -> str:
    # Simulates JSON-RPC execution on MCP Server
    appointment_id = f"APT-{abs(hash(patient_name)) % 10000}"
    return (
        f"[MCP Response: Clinic Server] Appointment {appointment_id} Confirmed! "
        f"Patient: {patient_name} | Department: {department.upper()} | Slot: Tomorrow ({preferred_time}) | Room: 304B"
    )

# 3. Adapt MCP Spec into LangChain Tool
mcp_adapted_tool = MCPToolAdapter.convert_mcp_to_langchain(
    mcp_schema=mcp_clinic_appointment_spec,
    execute_fn=handle_mcp_clinic_rpc
)

print(f"Successfully converted MCP tool: {mcp_adapted_tool.name}")
print(f"Tool Description: {mcp_adapted_tool.description}")

# 4. Bind MCP Tool to LangChain ReAct Agent and Execute
mcp_agent_tools = [mcp_adapted_tool]
mcp_agent = create_agent(
    model=llm,
    tools=mcp_agent_tools,
    system_prompt=(
        "You are an automated hospital scheduling assistant. Use the MCP clinic appointment tool "
        "to book patient appointments accurately based on user requests."
    )
)

query_mcp = "Book a pediatric clinic appointment for patient 'Leo Messi' for tomorrow morning using the MCP tool."
print("\n--- Executing Agent with Model Context Protocol (MCP) Tool ---")
result_mcp = mcp_agent.invoke({"messages": [("user", query_mcp)]})
print("\n--- MCP Agent Final Answer ---\n", result_mcp["messages"][-1].content)

### Summary & Key Engineering Takeaways

| Agent Pattern | Core Mechanics | Applied Domain |
| :--- | :--- | :--- |
| **Agent 1: Simple ReAct** | `create_agent` + `@tool` functions + message state | **Customer Support**: Order status lookup & restocking fee calculations. |
| **Agent 2: Resilient ReAct** | Native `@tool` + `duckduckgo_search` + `handle_tool_error=True` + `ToolException` | **Real Estate Finder**: Self-correcting rental database queries & affordability evaluations. |
| **Agent 3: Structured Conversational** | Multi-argument Pydantic schemas (`args_schema`) + message history | **Healthcare Clinic**: Safe pediatric dosage calculations & multi-turn allergy conflict checking. |
| **MCP Tool Integration** | Standardized JSON-RPC protocol + `MCPToolAdapter` | **Clinical Scheduling**: Seamlessly bridging standardized MCP server tools into LangChain. |